In [ ]:
# syracuse_snow_project.py

import requests
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime

# -----------------------------
# CONFIG
# -----------------------------
LATITUDE = 43.0481
LONGITUDE = -76.1474
CITY_NAME = "Syracuse, NY"

START_DATE = "2023-11-01"
END_DATE = "2024-03-31"

# -----------------------------
# FETCH DATA
# -----------------------------
def fetch_syracuse_snow_data(start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetch daily weather and snowfall data for Syracuse using Open-Meteo archive API.
    """
    url = "https://archive-api.open-meteo.com/v1/archive"

    params = {
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "start_date": start_date,
        "end_date": end_date,
        "daily": ",".join([
            "temperature_2m_max",
            "temperature_2m_min",
            "temperature_2m_mean",
            "precipitation_sum",
            "rain_sum",
            "snowfall_sum",
            "snow_depth_max",
            "weathercode"
        ]),
        "timezone": "America/New_York"
    }

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()
    data = response.json()

    if "daily" not in data:
        raise ValueError("Daily weather data not found in API response.")

    df = pd.DataFrame(data["daily"])
    df["time"] = pd.to_datetime(df["time"])
    df.rename(columns={"time": "date"}, inplace=True)

    return df


# -----------------------------
# FEATURE ENGINEERING
# -----------------------------
def build_snow_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add snow-related analytics columns.
    """
    df = df.copy()

    # Basic flags
    df["is_snow_day"] = df["snowfall_sum"].fillna(0) > 0
    df["is_heavy_snow_day"] = df["snowfall_sum"].fillna(0) >= 5  # mm threshold

    # Rolling features
    df["snowfall_7d"] = df["snowfall_sum"].fillna(0).rolling(7, min_periods=1).sum()
    df["precip_7d"] = df["precipitation_sum"].fillna(0).rolling(7, min_periods=1).sum()

    # Calendar features
    df["month"] = df["date"].dt.month
    df["day_name"] = df["date"].dt.day_name()
    df["week"] = df["date"].dt.isocalendar().week.astype(int)

    # Temperature indicator
    df["freezing_day"] = df["temperature_2m_mean"].fillna(99) <= 0

    # Simple severity score
    df["snow_severity_score"] = (
        df["snowfall_sum"].fillna(0) * 0.6
        + df["snow_depth_max"].fillna(0) * 0.3
        + df["precipitation_sum"].fillna(0) * 0.1
    )

    return df


# -----------------------------
# SUMMARY METRICS
# -----------------------------
def print_summary(df: pd.DataFrame) -> None:
    total_snowfall = df["snowfall_sum"].fillna(0).sum()
    snow_days = df["is_snow_day"].sum()
    heavy_snow_days = df["is_heavy_snow_day"].sum()
    max_snow_depth = df["snow_depth_max"].fillna(0).max()
    cold_days = df["freezing_day"].sum()

    print("\n--- Syracuse Snow Project Summary ---")
    print(f"City: {CITY_NAME}")
    print(f"Date Range: {df['date'].min().date()} to {df['date'].max().date()}")
    print(f"Total Snowfall (mm): {total_snowfall:.2f}")
    print(f"Snow Days: {snow_days}")
    print(f"Heavy Snow Days: {heavy_snow_days}")
    print(f"Max Snow Depth (cm): {max_snow_depth:.2f}")
    print(f"Freezing Days: {cold_days}")


# -----------------------------
# VISUALIZATIONS
# -----------------------------
def plot_daily_snowfall(df: pd.DataFrame) -> None:
    plt.figure(figsize=(14, 6))
    plt.bar(df["date"], df["snowfall_sum"].fillna(0))
    plt.title(f"Daily Snowfall in {CITY_NAME}")
    plt.xlabel("Date")
    plt.ylabel("Snowfall (mm)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


def plot_snow_depth(df: pd.DataFrame) -> None:
    plt.figure(figsize=(14, 6))
    plt.plot(df["date"], df["snow_depth_max"].fillna(0))
    plt.title(f"Daily Maximum Snow Depth in {CITY_NAME}")
    plt.xlabel("Date")
    plt.ylabel("Snow Depth (cm)")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


def plot_temperature_vs_snowfall(df: pd.DataFrame) -> None:
    plt.figure(figsize=(10, 6))
    plt.scatter(
        df["temperature_2m_mean"],
        df["snowfall_sum"].fillna(0),
        alpha=0.7
    )
    plt.title("Mean Temperature vs Snowfall")
    plt.xlabel("Mean Temperature (°C)")
    plt.ylabel("Snowfall (mm)")
    plt.tight_layout()
    plt.show()


def plot_weekly_snowfall(df: pd.DataFrame) -> None:
    weekly_df = df.groupby("week", as_index=False)["snowfall_sum"].sum()

    plt.figure(figsize=(12, 6))
    plt.plot(weekly_df["week"], weekly_df["snowfall_sum"], marker="o")
    plt.title(f"Weekly Snowfall Trend in {CITY_NAME}")
    plt.xlabel("ISO Week Number")
    plt.ylabel("Snowfall (mm)")
    plt.tight_layout()
    plt.show()


# -----------------------------
# EXPORT CLEAN DATA
# -----------------------------
def save_dataset(df: pd.DataFrame, file_name: str = "syracuse_snow_data.csv") -> None:
    df.to_csv(file_name, index=False)
    print(f"\nDataset saved as: {file_name}")


# -----------------------------
# MAIN
# -----------------------------
def main():
    try:
        raw_df = fetch_syracuse_snow_data(START_DATE, END_DATE)
        snow_df = build_snow_features(raw_df)

        print_summary(snow_df)
        save_dataset(snow_df)

        plot_daily_snowfall(snow_df)
        plot_snow_depth(snow_df)
        plot_temperature_vs_snowfall(snow_df)
        plot_weekly_snowfall(snow_df)

        print("\nFirst 5 rows:")
        print(snow_df.head())

    except Exception as e:
        print(f"Error while running the Syracuse snow project: {e}")


if __name__ == "__main__":
    main()